# Week 12, Day 4: Project: finish and test

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w012d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('students.csv', 'w', encoding='utf-8') as f:
    f.write('id,name,class\nS01,Ada Obi,3A\nS02,Bo Musa,3A\nS03,Cy Bello,3B\nS04,Di Eze,3B\nS05,Ed Okoro,3A\nS06,Fe Ade,3B\n')
with open('scores.csv', 'w', encoding='utf-8') as f:
    f.write('student,subject,score\nS01,maths,93\nS01,english,71\nS01,science,88\nS02,maths,41\nS02,english,88\nS02,science,57\nS03,maths,78\nS03,english,64\nS03,science,\nS04,maths,55\nS04,english,90\nS04,science,72\nS05,maths,67\nS05,english,abc\nS05,science,49\nS07,maths,80\nS02,maths,45\nS06,maths,101\nS06,english,58\nS06,science,61\nS06,art,70\n')

### Exercise: Step 6: class report

Write `class_report(summary, class_name)` that returns the report for one class as a string, lines joined with `"\n"`:

- a title line `Class 3B`;
- a header: `Pos` right-aligned in 3, two spaces, `Name` left-aligned in 12, `Average` right-aligned in 8, two spaces, `Grade`;
- one line per student in that class, ranked by average (highest first), equal averages by name, and students with no average last; positions count from 1; the average is shown with `show` (given), right-aligned in 8;
- a last line `Class average: 67.6`: the mean of the students' averages that exist, rounded to 1 place and shown with `show`.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def load_scores(path, students):
    scores = {sid: {} for sid in students}
    problems = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for line, row in enumerate(reader, start=2):
            sid, subject = row["student"].strip(), row["subject"].strip().lower()
            try:
                if sid not in students:
                    raise ValueError(f"unknown student {sid!r}")
                if subject not in SUBJECTS:
                    raise ValueError(f"unknown subject {subject!r}")
                if subject in scores[sid]:
                    raise ValueError(f"duplicate {subject} score for {sid}")
                scores[sid][subject] = check_score(row["score"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
    return scores, problems

def average(subject_scores):
    if not subject_scores:
        return None
    return round(sum(subject_scores.values()) / len(subject_scores), 1)

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

def summarise(students, scores):
    rows = []
    for sid, info in students.items():
        avg = average(scores[sid])
        rows.append({"id": sid, "name": info["name"], "class": info["class"], "average": avg,
                     "grade": grade(avg), "missing": [s for s in SUBJECTS if s not in scores[sid]]})
    return sorted(rows, key=lambda r: (r["class"], r["name"]))

def show(avg):
    return "-" if avg is None else f"{avg:.1f}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def data():\n    s = load_students(\"students.csv\")\n    sc, p = load_scores(\"scores.csv\", s)\n    return summarise(s, sc)\ndef test_3B():\n    \"class 3B\"\n    assert class_report(data(), \"3B\") == 'Class 3B\\nPos  Name         Average  Grade\\n  1  Di Eze          72.3  A\\n  2  Cy Bello        71.0  A\\n  3  Fe Ade          59.5  C\\nClass average: 67.6', \"Expected:\\n\" + 'Class 3B\\nPos  Name         Average  Grade\\n  1  Di Eze          72.3  A\\n  2  Cy Bello        71.0  A\\n  3  Fe Ade          59.5  C\\nClass average: 67.6'\ndef test_3A():\n    \"class 3A\"\n    assert class_report(data(), \"3A\") == 'Class 3A\\nPos  Name         Average  Grade\\n  1  Ada Obi         84.0  A\\n  2  Bo Musa         62.0  B\\n  3  Ed Okoro        58.0  C\\nClass average: 68.0', \"Expected:\\n\" + 'Class 3A\\nPos  Name         Average  Grade\\n  1  Ada Obi         84.0  A\\n  2  Bo Musa         62.0  B\\n  3  Ed Okoro        58.0  C\\nClass average: 68.0'\ndef test_none():\n    \"a student with no average\"\n    rows = [{\"id\": \"X\", \"name\": \"Zed\", \"class\": \"9\", \"average\": None, \"grade\": \"-\", \"missing\": []},\n            {\"id\": \"Y\", \"name\": \"Amy\", \"class\": \"9\", \"average\": 50.0, \"grade\": \"C\", \"missing\": []}]\n    assert class_report(rows, \"9\").splitlines()[2:] == [\"  1  Amy             50.0  C\", \"  2  Zed                -  -\", \"Class average: 50.0\"]")

### Exercise: Step 7: report cards

Write `report_card(row, subject_scores)` that returns the card for one summary row and that student's scores, ending with a newline:

- `Report card: NAME (ID), class CLASS`, then a blank line;
- one line per subject in `SUBJECTS` order: the subject with a capital first letter, left-aligned in 10, then the score right-aligned in 4, or `-` if there isn't one;
- a blank line, `Average: ` and the average shown with `show`, and `Grade: ` and the grade.

Then write `write_cards(summary, scores, folder="reports")` that makes the folder if needed, writes `folder/ID.txt` for every row, and returns how many it wrote.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def load_scores(path, students):
    scores = {sid: {} for sid in students}
    problems = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for line, row in enumerate(reader, start=2):
            sid, subject = row["student"].strip(), row["subject"].strip().lower()
            try:
                if sid not in students:
                    raise ValueError(f"unknown student {sid!r}")
                if subject not in SUBJECTS:
                    raise ValueError(f"unknown subject {subject!r}")
                if subject in scores[sid]:
                    raise ValueError(f"duplicate {subject} score for {sid}")
                scores[sid][subject] = check_score(row["score"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
    return scores, problems

def average(subject_scores):
    if not subject_scores:
        return None
    return round(sum(subject_scores.values()) / len(subject_scores), 1)

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

def summarise(students, scores):
    rows = []
    for sid, info in students.items():
        avg = average(scores[sid])
        rows.append({"id": sid, "name": info["name"], "class": info["class"], "average": avg,
                     "grade": grade(avg), "missing": [s for s in SUBJECTS if s not in scores[sid]]})
    return sorted(rows, key=lambda r: (r["class"], r["name"]))

def show(avg):
    return "-" if avg is None else f"{avg:.1f}"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def data():\n    s = load_students(\"students.csv\")\n    sc, p = load_scores(\"scores.csv\", s)\n    return summarise(s, sc), sc\ndef test_card():\n    \"report_card\"\n    rows, sc = data()\n    assert report_card(rows[0], sc[\"S01\"]) == 'Report card: Ada Obi (S01), class 3A\\n\\nMaths       93\\nEnglish     71\\nScience     88\\n\\nAverage: 84.0\\nGrade: A\\n', \"Expected:\\n\" + 'Report card: Ada Obi (S01), class 3A\\n\\nMaths       93\\nEnglish     71\\nScience     88\\n\\nAverage: 84.0\\nGrade: A\\n'\n    assert report_card(rows[2], sc[\"S05\"]) == 'Report card: Ed Okoro (S05), class 3A\\n\\nMaths       67\\nEnglish      -\\nScience     49\\n\\nAverage: 58.0\\nGrade: C\\n', \"Expected:\\n\" + 'Report card: Ed Okoro (S05), class 3A\\n\\nMaths       67\\nEnglish      -\\nScience     49\\n\\nAverage: 58.0\\nGrade: C\\n'\ndef test_files():\n    \"write_cards\"\n    reset_files()\n    rows, sc = data()\n    assert write_cards(rows, sc) == 6\n    assert read_file(\"reports/S01.txt\") == 'Report card: Ada Obi (S01), class 3A\\n\\nMaths       93\\nEnglish     71\\nScience     88\\n\\nAverage: 84.0\\nGrade: A\\n'\n    assert [f for f in files_now() if f.startswith(\"reports/\")] == [\"reports/S01.txt\", \"reports/S02.txt\", \"reports/S03.txt\", \"reports/S04.txt\", \"reports/S05.txt\", \"reports/S06.txt\"]\n    write_cards(rows[:1], sc, folder=\"out/term1\")\n    assert read_file(\"out/term1/S01.txt\") == 'Report card: Ada Obi (S01), class 3A\\n\\nMaths       93\\nEnglish     71\\nScience     88\\n\\nAverage: 84.0\\nGrade: A\\n'")

### Exercise: Step 8: top scorers

Write `top_scorers(students, scores)` that returns a dictionary from each subject (in `SUBJECTS` order, leaving out any subject nobody has a score for) to a tuple: the highest score, and the sorted list of names of everyone who got it.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def load_scores(path, students):
    scores = {sid: {} for sid in students}
    problems = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for line, row in enumerate(reader, start=2):
            sid, subject = row["student"].strip(), row["subject"].strip().lower()
            try:
                if sid not in students:
                    raise ValueError(f"unknown student {sid!r}")
                if subject not in SUBJECTS:
                    raise ValueError(f"unknown subject {subject!r}")
                if subject in scores[sid]:
                    raise ValueError(f"duplicate {subject} score for {sid}")
                scores[sid][subject] = check_score(row["score"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
    return scores, problems

def average(subject_scores):
    if not subject_scores:
        return None
    return round(sum(subject_scores.values()) / len(subject_scores), 1)

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

def summarise(students, scores):
    rows = []
    for sid, info in students.items():
        avg = average(scores[sid])
        rows.append({"id": sid, "name": info["name"], "class": info["class"], "average": avg,
                     "grade": grade(avg), "missing": [s for s in SUBJECTS if s not in scores[sid]]})
    return sorted(rows, key=lambda r: (r["class"], r["name"]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_file():\n    \"the files\"\n    s = load_students(\"students.csv\")\n    sc, p = load_scores(\"scores.csv\", s)\n    assert top_scorers(s, sc) == {'maths': (93, ['Ada Obi']), 'english': (90, ['Di Eze']), 'science': (88, ['Ada Obi'])}\ndef test_ties():\n    \"ties and gaps\"\n    s = {\"A\": {\"name\": \"Zed\", \"class\": \"1\"}, \"B\": {\"name\": \"Amy\", \"class\": \"1\"}}\n    sc = {\"A\": {\"maths\": 80}, \"B\": {\"maths\": 80, \"english\": 50}}\n    assert top_scorers(s, sc) == {\"maths\": (80, [\"Amy\", \"Zed\"]), \"english\": (50, [\"Amy\"])}")

### Exercise: Step 9: checks that catch mistakes

Write `run_checks()`, a function that uses `assert` to check `grade` and `check_score`. It must pass with the correct versions given in the starter, and fail (raise an `AssertionError`, or any other error) for each of six broken versions the checker will swap in, one at a time. You don't see the broken versions: think about what could go wrong. For example:

- a boundary written with `>` instead of `>=`;
- a grade missing, or `None` handled wrongly;
- `check_score` accepting a score over 100, or rejecting 0 or 100;
- `check_score` raising the wrong message, or not raising at all.

To check that something raises, use `try`, `except ValueError as e` (checking `str(e)`), and `else: raise AssertionError(...)`.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

def run_checks():
    ___

run_checks()
print("All checks passed")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def raise_on(text, message):\n    def broken(t):\n        raise ValueError(message)\n    return broken\nBROKEN_GRADE = {\n    \"70 graded B (> instead of >=)\": lambda a: \"-\" if a is None else (\"A\" if a > 70 else \"B\" if a >= 60 else \"C\" if a >= 50 else \"D\" if a >= 45 else \"E\" if a >= 40 else \"F\"),\n    \"no E grade\": lambda a: \"-\" if a is None else (\"A\" if a >= 70 else \"B\" if a >= 60 else \"C\" if a >= 50 else \"D\" if a >= 45 else \"F\"),\n    \"None gives F\": lambda a: \"F\" if a is None else (\"A\" if a >= 70 else \"B\" if a >= 60 else \"C\" if a >= 50 else \"D\" if a >= 45 else \"E\" if a >= 40 else \"F\"),\n}\ndef _ok_score(text):\n    text = text.strip()\n    if not text:\n        raise ValueError(\"missing score\")\n    if not text.isdigit():\n        raise ValueError(f\"not a whole number: {text!r}\")\n    if int(text) > 100:\n        raise ValueError(f\"out of range: {int(text)}\")\n    return int(text)\ndef _accepts_101(text):\n    text = text.strip()\n    if not text:\n        raise ValueError(\"missing score\")\n    if not text.isdigit():\n        raise ValueError(f\"not a whole number: {text!r}\")\n    if int(text) > 101:\n        raise ValueError(f\"out of range: {int(text)}\")\n    return int(text)\ndef _rejects_0(text):\n    if text.strip() == \"0\":\n        raise ValueError(\"out of range: 0\")\n    return _ok_score(text)\ndef _no_missing(text):\n    return int(text)\nBROKEN_SCORE = {\"accepts 101\": _accepts_101, \"rejects 0\": _rejects_0, \"empty text gives Python's own error\": _no_missing}\ndef test_passes():\n    \"passes with the correct functions\"\n    run_checks()\ndef caught(name, fn, which):\n    g = run_checks.__globals__\n    old = g[which]\n    g[which] = fn\n    try:\n        run_checks()\n    except Exception:\n        return True\n    finally:\n        g[which] = old\n    return False\ndef test_catches():\n    \"catches the broken versions\"\n    missed = [n for n, f in BROKEN_GRADE.items() if not caught(n, f, \"grade\")]\n    missed += [n for n, f in BROKEN_SCORE.items() if not caught(n, f, \"check_score\")]\n    assert not missed, \"These broken versions got past your checks: \" + \"; \".join(missed)")

> **Checks are worth writing.** Your checks just caught six realistic bugs that a quick test by hand would probably miss. In Phase 2 you'll meet **pytest**, which turns functions like `run_checks` into a proper test suite that runs every check, reports every failure, and runs in a second whenever you change your code.

### Exercise: Step 10: the gradebook program

Write `respond(command, students, scores, problems)`, which returns the text for one command:

- `classes`: each class with its number of students, sorted, like `3A (3 students), 3B (3 students)` (`1 student` for one);
- `report CLASS`: the class report, or `No class CLASS`;
- `student ID`: that student's report card without its final newline, accepting the id in any case, or `No student ID` (as typed);
- `problems`: the problems, one per line, or `No problems`;
- `cards`: writes the report cards to `reports` and returns `Wrote 6 report cards to reports/`;
- `top`: one line per subject, like `maths: 93 Ada Obi` (several names joined with `", "`);
- anything else: `Unknown command. ` followed by `HELP`.

Then write `main()`, which loads the files, prints `Gradebook: 6 students, 15 scores, 6 problems` (with the real counts) and `HELP`, then reads commands with the prompt `> ` (stripped) until `quit`, printing each response. The last line of the program calls `main()`.

In [ ]:
import csv
SUBJECTS = ["maths", "english", "science"]
BOUNDARIES = [(70, "A"), (60, "B"), (50, "C"), (45, "D"), (40, "E")]

def load_students(path):
    students = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            students[row["id"].strip()] = {"name": row["name"].strip(), "class": row["class"].strip()}
    return students

def check_score(text):
    text = text.strip()
    if not text:
        raise ValueError("missing score")
    if not text.isdigit():
        raise ValueError(f"not a whole number: {text!r}")
    score = int(text)
    if score > 100:
        raise ValueError(f"out of range: {score}")
    return score

def load_scores(path, students):
    scores = {sid: {} for sid in students}
    problems = []
    with open(path, newline="") as f:
        reader = csv.DictReader(f)
        for line, row in enumerate(reader, start=2):
            sid, subject = row["student"].strip(), row["subject"].strip().lower()
            try:
                if sid not in students:
                    raise ValueError(f"unknown student {sid!r}")
                if subject not in SUBJECTS:
                    raise ValueError(f"unknown subject {subject!r}")
                if subject in scores[sid]:
                    raise ValueError(f"duplicate {subject} score for {sid}")
                scores[sid][subject] = check_score(row["score"])
            except ValueError as e:
                problems.append(f"line {line}: {e}")
    return scores, problems

def average(subject_scores):
    if not subject_scores:
        return None
    return round(sum(subject_scores.values()) / len(subject_scores), 1)

def grade(avg):
    if avg is None:
        return "-"
    for lowest, letter in BOUNDARIES:
        if avg >= lowest:
            return letter
    return "F"

def summarise(students, scores):
    rows = []
    for sid, info in students.items():
        avg = average(scores[sid])
        rows.append({"id": sid, "name": info["name"], "class": info["class"], "average": avg,
                     "grade": grade(avg), "missing": [s for s in SUBJECTS if s not in scores[sid]]})
    return sorted(rows, key=lambda r: (r["class"], r["name"]))

def show(avg):
    return "-" if avg is None else f"{avg:.1f}"

def class_report(summary, class_name):
    rows = [r for r in summary if r["class"] == class_name]
    ranked = sorted(rows, key=lambda r: (r["average"] is None, -(r["average"] or 0), r["name"]))
    lines = [f"Class {class_name}", f"{'Pos':>3}  {'Name':<12}{'Average':>8}  Grade"]
    for pos, r in enumerate(ranked, start=1):
        lines.append(f"{pos:>3}  {r['name']:<12}{show(r['average']):>8}  {r['grade']}")
    known = [r["average"] for r in rows if r["average"] is not None]
    lines.append(f"Class average: {show(round(sum(known) / len(known), 1) if known else None)}")
    return "\n".join(lines)

def report_card(row, subject_scores):
    lines = [f"Report card: {row['name']} ({row['id']}), class {row['class']}", ""]
    for subject in SUBJECTS:
        score = subject_scores.get(subject)
        lines.append(f"{subject.capitalize():<10}{'-' if score is None else score:>4}")
    lines += ["", f"Average: {show(row['average'])}", f"Grade: {row['grade']}"]
    return "\n".join(lines) + "\n"

def top_scorers(students, scores):
    tops = {}
    for subject in SUBJECTS:
        marks = [(s[subject], students[sid]["name"]) for sid, s in scores.items() if subject in s]
        if marks:
            best = max(m for m, _ in marks)
            tops[subject] = (best, sorted(n for m, n in marks if m == best))
    return tops

HELP = "Commands: classes, report CLASS, student ID, problems, cards, top, quit"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_session():\n    \"a whole session\"\n    expect_output(\"Gradebook: 6 students, 15 scores, 6 problems\\nCommands: classes, report CLASS, student ID, problems, cards, top, quit\\n> classes\\n3A (3 students), 3B (3 students)\\n> report 3B\\nClass 3B\\nPos  Name         Average  Grade\\n  1  Di Eze          72.3  A\\n  2  Cy Bello        71.0  A\\n  3  Fe Ade          59.5  C\\nClass average: 67.6\\n> student s05\\nReport card: Ed Okoro (S05), class 3A\\n\\nMaths       67\\nEnglish      -\\nScience     49\\n\\nAverage: 58.0\\nGrade: C\\n> top\\nmaths: 93 Ada Obi\\nenglish: 90 Di Eze\\nscience: 88 Ada Obi\\n> report 9Z\\nNo class 9Z\\n> student S99\\nNo student S99\\n> problems\\nline 10: missing score\\nline 15: not a whole number: 'abc'\\nline 17: unknown student 'S07'\\nline 18: duplicate maths score for S02\\nline 19: out of range: 101\\nline 22: unknown subject 'art'\\n> dance\\nUnknown command. Commands: classes, report CLASS, student ID, problems, cards, top, quit\\n> quit\", inputs=['classes', 'report 3B', 'student s05', 'top', 'report 9Z', 'student S99', 'problems', 'dance', 'quit'], echo=True)\ndef test_cards():\n    \"cards\"\n    reset_files()\n    s = load_students(\"students.csv\")\n    sc, p = load_scores(\"scores.csv\", s)\n    assert respond(\"cards\", s, sc, p) == \"Wrote 6 report cards to reports/\"\n    assert read_file(\"reports/S04.txt\") is not None\ndef test_no_problems():\n    \"no problems\"\n    s = {\"A\": {\"name\": \"Zed\", \"class\": \"1\"}}\n    assert respond(\"problems\", s, {\"A\": {}}, []) == \"No problems\"\n    assert respond(\"classes\", s, {\"A\": {}}, []) == \"1 (1 student)\"")

> **What would you change?.** Write down two things you'd do differently if you built it again, and one feature a teacher might ask for next (a fourth subject? weighted averages? a "most improved" list?). For each feature, which functions would have to change? If the answer is "only one or two", the design is doing its job.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

After yesterday, the gradebook has these working functions: `load_students`, `check_score`, `load_scores`, `average`, `grade` and `summarise`. `summarise` gives a list of rows like this, sorted by class and name:

```
{"id": "S05", "name": "Ed Okoro", "class": "3A", "average": 58.0, "grade": "C", "missing": ["english"]}
```

Everything today builds on that list. As yesterday, each step's starting code includes model versions of all the earlier functions.

## Step 6: class reports

A teacher wants each class ranked, best average first:

```
Class 3B
Pos  Name         Average  Grade
  1  Di Eze          72.3  A
  2  Cy Bello        71.0  A
  3  Fe Ade          59.5  C
Class average: 67.6
```

## Step 7: report cards

Each student gets a report card, saved as a text file named by their id in a `reports` folder:

```
Report card: Ed Okoro (S05), class 3A

Maths       67
English      -
Science     49

Average: 58.0
Grade: C
```

## Step 8: top scorers

## Step 9: checking your work

The functions most likely to hide a mistake are the ones full of boundaries: `grade` (is 70 an A or a B?) and `check_score` (is 100 allowed? is 0?). In Week 7 you checked functions with `assert`. Now write checks that would **catch** the mistakes someone is most likely to make in these two.

## Step 10: the program

The last step is the part a teacher actually uses: a command loop. Here's a session:

```
Gradebook: 6 students, 15 scores, 6 problems
Commands: classes, report CLASS, student ID, problems, cards, top, quit
> classes
3A (3 students), 3B (3 students)
> report 3B
Class 3B
Pos  Name         Average  Grade
  1  Di Eze          72.3  A
  2  Cy Bello        71.0  A
  3  Fe Ade          59.5  C
Class average: 67.6
> student s05
Report card: Ed Okoro (S05), class 3A

Maths       67
English      -
Science     49

Average: 58.0
Grade: C
> top
maths: 93 Ada Obi
english: 90 Di Eze
science: 88 Ada Obi
> report 9Z
No class 9Z
> student S99
No student S99
> problems
line 10: missing score
line 15: not a whole number: 'abc'
line 17: unknown student 'S07'
line 18: duplicate maths score for S02
line 19: out of range: 101
line 22: unknown subject 'art'
> dance
Unknown command. Commands: classes, report CLASS, student ID, problems, cards, top, quit
> quit
```

## Looking back

You've built a complete program of about 150 lines. Before moving on, look at how it's put together:

- **Every function does one thing**, and most are under 15 lines. When a teacher asks for a new command, you add one function and one branch in `respond`; nothing else changes.
- **Reading, calculating and presenting are separate.** `load_scores` knows nothing about reports; `class_report` knows nothing about files. You could replace the CSV files with a database, or the text reports with a web page, and the middle would stay the same.
- **Bad data is expected, reported and never fatal.** Every problem has a line number and a reason.
- **The risky parts are checked.** Your checks would tell you at once if someone "tidied up" `grade` and broke a boundary.

## Quiz

## Before you go

Run your finished program, try every command, and try a few things a careless user might type. Fix anything that crashes. Tomorrow is the Phase 1 exam: get a good night's sleep, and have a glance at your revision list in the morning.